# Maia Lite 0.3.0 — LoRA experimental com corpus de até 200 mil diálogos
**L4 / A100 · checkpoint base 300k · treino sem validação periódica.** Não usa adaptadores v2.2/v2.3. Faz checagens estruturais mínimas, salva checkpoints e métricas. O orçamento de 20 CUs / 5h é uma hipótese a medir, **não um limite garantido**.

**Atenção:** corpus experimental sem auditoria jurídica e sem revisão humana. Não distribuir pesos/dados derivados antes de resolver licenças. O notebook não treina até `RUN_TRAINING=True`.

## 1. Ambiente e caminhos

In [ ]:
import os,json,math,time,random,hashlib,gc,sys,subprocess
from pathlib import Path
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(),'GPU necessária (L4 ou A100)'
SEED=20261120
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
DATA=ROOT/'datasets/maia_conversational_experiment_200k_v1/train.jsonl'
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
OUT=ROOT/'training_runs/maia_lite_0_3_0_sft_exp_200k_lora_v1'
for p in (DATA,BASE_CKPT):assert p.is_file(),f'Ausente: {p}'
assert TOKENIZER_DIR.is_dir()
print('GPU:',torch.cuda.get_device_name(0),'| dados:',DATA)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4 | dados: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets/maia_conversational_experiment_200k_v1/train.jsonl


## 2. Hiperparâmetros

In [ ]:
RANK=16;ALPHA=32;LORA_DROPOUT=0.05
LR=5e-5;WEIGHT_DECAY=0.0
MAX_LENGTH=768;MICRO_BATCH=1;GRAD_ACCUM=16;EPOCHS=1
MAX_STEPS=None   # Exemplo: 20 para teste de fumaça
SAVE_EVERY=250;LOG_EVERY=20;MAX_GRAD_NORM=1.0
RUN_TRAINING=True  # altere para True quando quiser iniciar
MAX_EXAMPLES=None  # limite opcional, ex.: 5000
assert MAX_LENGTH<=2048


## 3. Arquitetura idêntica ao checkpoint base

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## 4. Carregamento do checkpoint e LoRA

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size
assert tokenizer.eos_token_id is not None
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False) # somente checkpoint próprio/confiável
assert 'model' in checkpoint and int(checkpoint.get('step',-1))==300000
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint;gc.collect()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)

for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,RANK,ALPHA,LORA_DROPOUT)
    block.proj=LoRALinear(block.proj,RANK,ALPHA,LORA_DROPOUT)
trainable=[p for p in model.parameters() if p.requires_grad]
print('LoRA trainable parameters:',sum(p.numel() for p in trainable))
assert len(trainable)==cfg.n_layer*2*2


LoRA trainable parameters: 3538944


## 5. Converter conversas em pares de treinamento
Cada turno do assistente gera um exemplo com o histórico anterior como prompt. O texto do usuário não recebe loss. Conversas excessivamente longas são descartadas (sem truncamento silencioso da resposta). O formato de prompt é **novo** e deve ser usado também na inferência.

In [ ]:
def iter_jsonl(path):
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():yield json.loads(line)

def turn_pairs(rec):
    history=[]
    for m in rec['messages']:
        role=m['role'];txt=m['content'].strip()
        if role=='user':history.append('Usuário: '+txt)
        elif role=='assistant':
            prompt='\n'.join(history)+'\nAssistente:'
            yield prompt,txt,rec.get('lang','unknown'),rec.get('id','')
            history.append('Assistente: '+txt)

def encode(prompt,answer,lang,rid):
    prefix=prompt+'\n'
    # Tokenização independente para garantir a fronteira de loss; compatível com tokenizer GPT2 BPE
    ids_prompt=tokenizer.encode(prefix,add_special_tokens=False)
    ids_answer=tokenizer.encode(answer,add_special_tokens=False)
    ids=ids_prompt+ids_answer+[tokenizer.eos_token_id]
    if len(ids)>MAX_LENGTH or len(ids_answer)<2 or len(ids_prompt)<2:return None
    x=ids[:-1];y=[-100]*(len(ids)-1)
    for i in range(len(y)):
        if i+1>=len(ids_prompt):y[i]=ids[i+1]
    return {'x':x,'y':y,'lang':lang,'id':rid}
train_data=[];drop=0;records=0
for rec in tqdm(iter_jsonl(DATA),desc='Tokenização'):
    records+=1
    for prompt,answer,lang,rid in turn_pairs(rec):
        item=encode(prompt,answer,lang,rid)
        if item:train_data.append(item)
        else:drop+=1
        if MAX_EXAMPLES and len(train_data)>=MAX_EXAMPLES:break
    if MAX_EXAMPLES and len(train_data)>=MAX_EXAMPLES:break
print('Conversas:',records,'pares:',len(train_data),'descartados:',drop)
assert train_data,'Nenhum par válido'
print('Exemplo tokens:',len(train_data[0]['x']),'tokens supervisionados:',sum(v!=-100 for v in train_data[0]['y']))


Tokenização: 0it [00:00, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1166 > 1024). Running this sequence through the model will result in indexing errors


Conversas: 73142 pares: 94862 descartados: 6369
Exemplo tokens: 618 tokens supervisionados: 589


## 6. Batch, smoke test e estimativa de passos

In [ ]:
def collate(items):
    L=max(len(r['x']) for r in items)
    x=torch.full((len(items),L),tokenizer.eos_token_id,dtype=torch.long)
    y=torch.full((len(items),L),-100,dtype=torch.long)
    for i,r in enumerate(items):
        l=len(r['x']);x[i,:l]=torch.tensor(r['x']);y[i,:l]=torch.tensor(r['y'])
    return x.cuda(),y.cuda()

probe=max(train_data,key=lambda r:len(r['x']))
model.train();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
x,y=collate([probe])
with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
loss=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100)
loss.backward();torch.cuda.synchronize()
print('Smoke test loss:',float(loss),'VRAM GiB:',round(torch.cuda.max_memory_allocated()/2**30,2))
for p in trainable:p.grad=None
del x,y,logits,loss;gc.collect();torch.cuda.empty_cache()
updates_per_epoch=math.ceil(len(train_data)/(MICRO_BATCH*GRAD_ACCUM))
planned_updates=updates_per_epoch*EPOCHS
if MAX_STEPS is not None:planned_updates=min(planned_updates,MAX_STEPS)
print('Updates planejados:',planned_updates)


Smoke test loss: 3.0160717964172363 VRAM GiB: 2.31
Updates planejados: 5929


## 7. Treinamento e retomada
Checkpoint completo inclui LoRA, otimizador, scheduler e RNG. A retomada ocorre no último **update completo**. A ordem de exemplos é determinada por época e seed. Sem validação periódica, `adapter_latest.pt` é o resultado; **não é um best-model**.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def lora_state():
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items() if k.endswith('.A') or k.endswith('.B')}
def restore_lora(state):
    with torch.no_grad():
        for name,p in model.named_parameters():
            if name in state:p.copy_(state[name].to(p.device,dtype=p.dtype))

optimizer=torch.optim.AdamW(trainable,lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s:max(0.1,1-s/max(planned_updates,1)))
CONFIG={'rank':RANK,'alpha':ALPHA,'dropout':LORA_DROPOUT,'lr':LR,'max_length':MAX_LENGTH,
 'micro_batch':MICRO_BATCH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'max_steps':MAX_STEPS,
 'max_examples':MAX_EXAMPLES,'seed':SEED,'data_sha256':sha256(DATA),'base_step':300000,
 'format':'Usuário:/Assistente:','train_pairs':len(train_data),'torch':torch.__version__}
OUT.mkdir(parents=True,exist_ok=True)
config_path=OUT/'config.json'
if config_path.exists():assert json.loads(config_path.read_text())==CONFIG,'Config mudou: use novo OUT'
else:config_path.write_text(json.dumps(CONFIG,indent=2,ensure_ascii=False))
ckpt=OUT/'adapter_latest.pt';start_update=0
if ckpt.exists():
    saved=torch.load(ckpt,map_location='cpu',weights_only=False)
    assert saved['config']==CONFIG
    restore_lora(saved['adapter']);optimizer.load_state_dict(saved['optimizer']);scheduler.load_state_dict(saved['scheduler'])
    random.setstate(saved['python_rng']);np.random.set_state(saved['numpy_rng'])
    torch.set_rng_state(saved['torch_rng']);torch.cuda.set_rng_state_all(saved['cuda_rng'])
    start_update=saved['update'];print('Retomada no update:',start_update)
    del saved

def save(update):
    state={'adapter':lora_state(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
      'update':update,'config':CONFIG,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
      'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
    tmp=OUT/'adapter_latest.tmp.pt';torch.save(state,tmp);os.replace(tmp,ckpt)
    print('Salvo update',update)


In [ ]:
if not RUN_TRAINING:
    print('Treinamento desativado. Para iniciar, defina RUN_TRAINING=True e execute novamente desde o início.')
else:
    assert start_update<=planned_updates
    model.train();optimizer.zero_grad(set_to_none=True)
    update=start_update;t0=time.time()
    progress=tqdm(total=planned_updates,initial=update,desc='LoRA exp 200k')
    try:
        for epoch in range(EPOCHS):
            if update>=planned_updates:break
            order=np.random.default_rng(SEED+epoch).permutation(len(train_data)).tolist()
            batches=[order[i:i+MICRO_BATCH] for i in range(0,len(order),MICRO_BATCH)]
            for local_update in range(math.ceil(len(batches)/GRAD_ACCUM)):
                global_update=epoch*updates_per_epoch+local_update
                if global_update<update:continue
                if update>=planned_updates:break
                group=batches[local_update*GRAD_ACCUM:(local_update+1)*GRAD_ACCUM]
                supervised=sum(sum(v!=-100 for v in train_data[j]['y']) for batch in group for j in batch)
                loss_acc=0.
                for ids in group:
                    x,y=collate([train_data[j] for j in ids])
                    with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(x)
                    loss_sum=F.cross_entropy(logits.float().reshape(-1,cfg.vocab_size),y.reshape(-1),ignore_index=-100,reduction='sum')
                    (loss_sum/max(supervised,1)).backward()
                    loss_acc+=float(loss_sum.detach())
                    del x,y,logits,loss_sum
                grad_norm=torch.nn.utils.clip_grad_norm_(trainable,MAX_GRAD_NORM)
                optimizer.step();scheduler.step();optimizer.zero_grad(set_to_none=True)
                update+=1;progress.update(1)
                if update%LOG_EVERY==0 or update==planned_updates:
                    elapsed=time.time()-t0
                    metrics={'update':update,'loss':loss_acc/max(supervised,1),'grad_norm':float(grad_norm),
                             'elapsed_s':elapsed,'updates_per_s':(update-start_update)/max(elapsed,1),
                             'lr':optimizer.param_groups[0]['lr']}
                    with (OUT/'metrics.jsonl').open('a') as f:f.write(json.dumps(metrics)+'\n')
                    print(metrics)
                if update%SAVE_EVERY==0 or update==planned_updates:save(update)
    except KeyboardInterrupt:
        print('Interrompido. Retome a partir do último checkpoint:',ckpt)
        raise
    finally:progress.close()


LoRA exp 200k:   0%|          | 0/5929 [00:00<?, ?it/s]

{'update': 20, 'loss': 2.5406800368932716, 'grad_norm': 0.25894737243652344, 'elapsed_s': 35.43667936325073, 'updates_per_s': 0.5643869673844443, 'lr': 4.9831337493675155e-05}
{'update': 40, 'loss': 2.6041410039082993, 'grad_norm': 0.33775654435157776, 'elapsed_s': 70.5247106552124, 'updates_per_s': 0.5671770876956252, 'lr': 4.9662674987350315e-05}
{'update': 60, 'loss': 3.0528196994974577, 'grad_norm': 0.3156087100505829, 'elapsed_s': 105.90275287628174, 'updates_per_s': 0.5665575102669287, 'lr': 4.949401248102547e-05}
{'update': 80, 'loss': 2.6898976135723696, 'grad_norm': 0.4172045886516571, 'elapsed_s': 141.25284337997437, 'updates_per_s': 0.5663602805134167, 'lr': 4.932534997470063e-05}
{'update': 100, 'loss': 2.605066519752543, 'grad_norm': 0.3718981146812439, 'elapsed_s': 176.68081951141357, 'updates_per_s': 0.5659923939482293, 'lr': 4.915668746837579e-05}
{'update': 120, 'loss': 2.1617244792757857, 'grad_norm': 0.35491472482681274, 'elapsed_s': 211.8269374370575, 'updates_per_s

## 8. Interpretação
- Observe `metrics.jsonl` e a taxa de updates/s; extrapole o custo real após 50–100 updates.
- Não confunda queda da loss de treino com melhoria conversacional.
- O formato de inferência deverá terminar em `\nAssistente:\n`.
- Este experimento não oferece avaliação qualitativa automática; faça perguntas PT/EN/ES após salvar o adaptador.
- Se interromper antes do primeiro checkpoint, os updates não salvos serão refeitos.